# LangChain Integration with Prompt Caching

This notebook demonstrates how to use prompt caching with LangChain's `ChatBedrockConverse` class. LangChain provides a `create_cache_point()` method that generates the `cachePoint` content block for you.

## What you'll learn

- `ChatBedrockConverse.create_cache_point()` for adding cache points to messages
- Integrating cache points into LCEL chains via `ChatPromptTemplate`
- Reading cache metrics from `usage_metadata`

## Prerequisites

- AWS credentials configured (default profile)
- Access to Claude Sonnet 4.6 on Amazon Bedrock
- `langchain-aws >= 0.2.12`, `boto3 >= 1.35.76`

In [ ]:
!pip install --upgrade --quiet langchain-aws boto3

In [ ]:
import json
import time

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL_ID = "global.anthropic.claude-sonnet-4-6-v1:0"
AWS_REGION = "us-west-2"

llm = ChatBedrockConverse(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
    max_tokens=1000
)

print(f"Model: {MODEL_ID}")
print(f"Region: {AWS_REGION}")

## Sample Document

Inline content exceeding the 1,024-token threshold for Claude Sonnet 4.6.

In [ ]:
DOCUMENT = """
The universe is a vast and mysterious expanse that has captivated human imagination for millennia. From the earliest civilizations who looked up at the night sky and wondered about the nature of the stars, to modern astronomers using sophisticated telescopes and spacecraft to explore distant galaxies, our quest to understand the cosmos continues unabated.

Our solar system, located in the Milky Way galaxy, is home to eight planets, numerous dwarf planets, and countless smaller objects including asteroids, comets, and meteoroids. The Sun, a middle-aged G-type main-sequence star, provides the energy that sustains life on Earth and influences the dynamics of all objects within its gravitational reach.

Mercury, the innermost planet, experiences extreme temperature variations due to its proximity to the Sun and lack of substantial atmosphere. Venus, often called Earth's twin due to its similar size, has a thick atmosphere composed primarily of carbon dioxide, creating a runaway greenhouse effect that makes it the hottest planet in our solar system. Earth, our home, is the only known planet to harbor life, with its unique combination of liquid water, moderate temperatures, and protective magnetic field.

Mars, the Red Planet, has long been a subject of fascination and speculation about the possibility of extraterrestrial life. Its rusty appearance comes from iron oxide prevalent on its surface. The planet features the largest volcano in the solar system, Olympus Mons, and a canyon system, Valles Marineris, that dwarfs the Grand Canyon.

Jupiter, the largest planet, is a gas giant composed primarily of hydrogen and helium. Its Great Red Spot, a persistent anticyclonic storm, has been observed for over 400 years. Jupiter's intense magnetic field and numerous moons, including the four Galilean satellites discovered by Galileo Galilei in 1610, make it a miniature solar system in its own right.

Saturn, famous for its spectacular ring system, is another gas giant with dozens of moons. Titan, its largest moon, has a thick atmosphere and liquid hydrocarbon lakes, making it one of the most intriguing bodies in the solar system for astrobiological research.

Uranus and Neptune, the ice giants, reside in the outer reaches of our solar system. Uranus rotates on its side, likely due to a massive impact early in its history. Neptune, the windiest planet, features storms with wind speeds exceeding 2,000 kilometers per hour.

Beyond Neptune lies the Kuiper Belt, a region populated by icy bodies including the dwarf planet Pluto. The New Horizons mission's flyby of Pluto in 2015 revealed a geologically active world with nitrogen glaciers and a hazy atmosphere. Even further out is the Oort Cloud, a hypothetical spherical shell of icy objects that may extend halfway to the nearest star.

Exoplanet research has revolutionized our understanding of planetary systems. The Kepler space telescope discovered thousands of planets orbiting other stars, revealing that planets are common throughout our galaxy. The James Webb Space Telescope observes in infrared to study the earliest galaxies and probe planetary atmospheres for signs of life.
"""

QUESTIONS = [
    "What are the ice giant planets and what makes them unique?",
    "What discoveries has the New Horizons mission made?",
    "How does Jupiter's Great Red Spot compare to storms on Earth?",
]

## 1. Direct Message Construction with `create_cache_point()`

Add a cache point directly in the `HumanMessage` content array. The document is cached, and subsequent queries with different questions reuse the cached prefix.

```python
HumanMessage(content=[
    {"type": "text", "text": document_content},
    ChatBedrockConverse.create_cache_point()
])
```

In [ ]:
def chat_with_cache(question):
    """Send a question about the document with prompt caching enabled."""
    messages = [
        SystemMessage(content="You are a helpful assistant that answers questions about documents."),
        HumanMessage(content=[
            {"type": "text", "text": f"Here is a document: <document>{DOCUMENT}</document>"},
            ChatBedrockConverse.create_cache_point()
        ]),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    return response

# Request 1 — cache write
print("Request 1 (cache write expected):")
r1 = chat_with_cache(QUESTIONS[0])
print(f"Response: {r1.content[:200]}...")
print(f"Usage: {json.dumps(r1.usage_metadata, indent=2, default=str)}")

time.sleep(1)

# Request 2 — cache read
print("\nRequest 2 (cache read expected):")
r2 = chat_with_cache(QUESTIONS[1])
print(f"Response: {r2.content[:200]}...")
print(f"Usage: {json.dumps(r2.usage_metadata, indent=2, default=str)}")

## 2. LCEL Chain with Cache Point

Integrate the cache point into a `ChatPromptTemplate` for reusable chain patterns.

In [ ]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions about documents."),
    ("human", [
        {"type": "text", "text": f"Here is a document: <document>{DOCUMENT}</document>"},
        ChatBedrockConverse.create_cache_point()
    ]),
    ("human", "{question}")
])

chain = template | llm | StrOutputParser()

# Run with multiple questions
for i, q in enumerate(QUESTIONS):
    start = time.time()
    result = chain.invoke({"question": q})
    elapsed = time.time() - start
    print(f"\nQ{i+1}: {q}")
    print(f"A: {result[:150]}...")
    print(f"Time: {elapsed:.2f}s")
    time.sleep(0.5)

## 3. Inspecting Cache Metrics

The `usage_metadata` on the response object includes `input_token_details` with cache-specific fields:
- `cache_creation`: tokens written to cache (first request)
- `cache_read`: tokens read from cache (subsequent requests)

In [ ]:
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=[
        {"type": "text", "text": f"Here is a document: <document>{DOCUMENT}</document>"},
        ChatBedrockConverse.create_cache_point()
    ]),
    HumanMessage(content="Summarize the key points about exoplanets.")
]

# First call — cache write
r1 = llm.invoke(messages)
print("First call usage_metadata:")
print(json.dumps(r1.usage_metadata, indent=2, default=str))

time.sleep(1)

# Second call — same prefix, different question
messages[-1] = HumanMessage(content="What is the most interesting moon in our solar system?")
r2 = llm.invoke(messages)
print("\nSecond call usage_metadata:")
print(json.dumps(r2.usage_metadata, indent=2, default=str))

## Conclusion

LangChain's `ChatBedrockConverse` integrates seamlessly with Bedrock prompt caching:

- `create_cache_point()` generates the correct `cachePoint` content block
- Works in both direct message construction and LCEL chain templates
- Cache metrics are available via `usage_metadata` on the response

For more details, see:
- [Amazon Bedrock Prompt Caching documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html)
- [LangChain ChatBedrockConverse documentation](https://python.langchain.com/docs/integrations/chat/bedrock/)